In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import pandas as pd
import numpy as np
import random
import pickle
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from tqdm import tqdm
from sklearn.metrics import r2_score
import seaborn as sns
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split


from data.data_loader_extend import EPCDataset, UMassDataset
from models.utils_extend import create_model, train_for_long_term_forecast, train_for_short_term_forecast, evaluate_for_long_term_forecast, evaluate_for_short_term_forecast


from explainers.utils_extend import get_explainer, unpack_eval_for_single

In [2]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
is_long_term_forecast = False

# dataset_name = 'epc'
dataset_name = 'umass'

if dataset_name == 'epc':
    file_path = 'data/final_data.csv'
    target_name = 'Global_active_power'
else:
    file_path = "data/umass/HomeA/HomeA_with_weather.csv"
    target_name = 'use_total'
    
def get_base_features(path):
    df = pd.read_csv(path)
    features = df.columns.tolist()
    return [f.lower() for f in features]


dataset_params = {
    # Long-term forecast
    'long_term_length' : 90*24,           # 입력 길이 (예: 90일치, 1시간 단위)
    'long_term_pred_length' : 256,        # 출력 길이 (예: 256시간)

    # Short-term forecast
    'short_term_length' : 30*24,          # 입력 길이 (예: 30일치)
    'short_term_pred_length' : 7*24,      # 출력 길이 (예: 7일치)

    # 단일 short-term (LSTM/GRU/CNN-LSTM 전용)
    'sequence_length' : 24*30,            # 입력 길이 (예: 30일치)
    'prediction_length' : 24              # 출력 길이 (예: 하루치)
}


base_features = get_base_features(file_path)
len(base_features)

48

In [4]:
def _resolve_filtered_list(filtered_features, which):
    """
    filtered_features가 dict({'long': [...], 'short': [...]}) 또는 list([...]) 또는 None일 수 있다.
    which: 'long' | 'short' | 'single'
    반환: list 또는 None
    """
    if filtered_features is None:
        return None
    if isinstance(filtered_features, dict):
        return filtered_features.get(which)
    if isinstance(filtered_features, (list, tuple)):
        return list(filtered_features)
    return None

def load_dataset(params,
                 dataset_type="epc",
                 file_path=None,
                 is_long_term_forecast=True,
                 target_name=None,
                 categorical_features=('icon',),
                 filtered_features=None):
    """
    params: {
        # long/short 모두 쓰는 경우
        "long_term_length": int,
        "long_term_pred_length": int,
        "short_term_length": int,
        "short_term_pred_length": int,

        # single 모드만 쓰는 경우
        "sequence_length": int,
        "prediction_length": int,
    }

    dataset_type: "epc" | "umass"
    target_name:  타깃 컬럼명 하나만 명시(Optional). None이면
                  EPC→Global_active_power, UMass→use_total, 없으면 첫 숫자형.
    categorical_features: UMass에서 임베딩 인덱스로 뽑을 범주형 컬럼 tuple
    """
    selected_features = {}

    # 어떤 클래스를 쓸지 선택
    if dataset_type.lower() == "epc":
        DatasetClass = EPCDataset
        is_umass = False
        ds_extra_kwargs = {}
    elif dataset_type.lower() == "umass":
        DatasetClass = UMassDataset
        is_umass = True
        # 여기서는 공통 kwargs만 넣고, filtered_features는 인스턴스 생성 시점에 주입
        ds_extra_kwargs = {
            "categorical_features": categorical_features,
        }
    else:
        raise ValueError(f"Unknown dataset_type: {dataset_type}")

    if is_long_term_forecast:
        # ----- Long -----
        ds_long = DatasetClass(
            file_path=file_path,
            sequence_length=params["long_term_length"],
            prediction_length=params["long_term_pred_length"],
            target_name=target_name,
            filtered_features=_resolve_filtered_list(filtered_features, "long"), 
            **ds_extra_kwargs
        )
        
        long_out = ds_long.load_data()
        target_scaler_long = ds_long.target_scaler
        
        # ----- Short -----
        ds_short = DatasetClass(
            file_path=file_path,
            sequence_length=params["short_term_length"],
            prediction_length=params["short_term_pred_length"],
            target_name=target_name,
            filtered_features=_resolve_filtered_list(filtered_features, "short"), 
            **ds_extra_kwargs
        )
        short_out = ds_short.load_data()
        target_scaler_short = ds_short.target_scaler

        target_scaler = (target_scaler_long, target_scaler_short)

        if is_umass:
            # UMass: 6개 반환
            Xn_tr_L, Xi_tr_L, y_tr_L, Xn_ev_L, Xi_ev_L, y_ev_L = long_out
            Xn_tr_S, Xi_tr_S, y_tr_S, Xn_ev_S, Xi_ev_S, y_ev_S = short_out

            train_data    = ((Xn_tr_L, Xi_tr_L), (Xn_tr_S, Xi_tr_S))
            train_targets = (y_tr_L, y_tr_S)
            eval_data     = ((Xn_ev_L, Xi_ev_L), (Xn_ev_S, Xi_ev_S))
            eval_targets  = (y_ev_L, y_ev_S)
        else:
            # EPC: 4개 반환
            train_long, train_targets_long, eval_long, eval_targets_long = long_out
            train_short, train_targets_short, eval_short, eval_targets_short = short_out

            train_data    = (train_long,  train_short)
            train_targets = (train_targets_long, train_targets_short)
            eval_data     = (eval_long,   eval_short)
            eval_targets  = (eval_targets_long, eval_targets_short)

        # auto-selected numeric feature names를 우선 제공
        get_names = lambda ds: (ds.get_numeric_feature_names()
                                if hasattr(ds, "get_numeric_feature_names")
                                else getattr(ds, "selected_features", []))
        selected_features["long"]  = get_names(ds_long)
        selected_features["short"] = get_names(ds_short)

    else:
        # ----- Single (short only) -----
        ds_short = DatasetClass(
            file_path=file_path,
            sequence_length=params["sequence_length"],
            prediction_length=params["prediction_length"],
            target_name=target_name,
            filtered_features=_resolve_filtered_list(filtered_features, "single"),  
            **ds_extra_kwargs
        )
        out = ds_short.load_data()
        target_scaler = ds_short.target_scaler

        if is_umass:
            Xn_tr, Xi_tr, y_tr, Xn_ev, Xi_ev, y_ev = out
            train_data    = (Xn_tr, Xi_tr)
            train_targets = y_tr
            eval_data     = (Xn_ev, Xi_ev)
            eval_targets  = y_ev
        else:
            train_seq, train_tgt, eval_seq, eval_tgt = out
            train_data,  train_targets = train_seq, train_tgt
            eval_data,   eval_targets  = eval_seq,  eval_tgt

        if hasattr(ds_short, "get_numeric_feature_names"):
            selected_features["single"] = ds_short.get_numeric_feature_names()
        else:
            selected_features["single"] = getattr(ds_short, "selected_features", [])

    return train_data, train_targets, eval_data, eval_targets, selected_features, target_scaler


def load_trained_model(
    params,
    is_long_term_forecast=True,
    oversample_eval=True,
    icon_embed_dim=8,
    path_suffix=None
):
    if params is None:
        raise ValueError("params must be provided")

    model_name    = params['model_name']
    hidden_size   = params['hidden_size']
    num_layers    = params['num_layers']
    dropout       = params['dropout']
    num_epochs    = params['num_epochs']
    batch_size    = params['batch_size']
    learning_rate = params['learning_rate']
    patience      = params['patience']
    mse_decay     = params.get('mse_decay', False)

    output_size = {
        'long'  : dataset_params.get('long_term_pred_length', dataset_params.get('prediction_length', 24)),
        'short' : dataset_params.get('short_term_pred_length', dataset_params.get('prediction_length', 24)),
        'single': dataset_params.get('prediction_length', 24),
    }

    mse_alpha, mse_beta = (0.3, 1.0) if mse_decay else (1.0, 1.0)

    icon_vocab_size = None

    if is_long_term_forecast:
        # EPC: train_data = (XL, XS)
        # UMass: train_data = ((XnL, XiL), (XnS, XiS))
        train_long, train_short = train_data
        is_umass = isinstance(train_long, (tuple, list)) and isinstance(train_short, (tuple, list))
    
        if is_umass:
            XnL, XiL = train_long
            XnS, XiS = train_short
            in_long  = XnL.shape[2]
            in_short = XnS.shape[2]
            icon_vocab_size = int(torch.max(torch.stack([XiL.max(), XiS.max()])).item()) + 1
        else:
            XL, XS = train_long, train_short
            in_long  = XL.shape[2]
            in_short = XS.shape[2]
            icon_vocab_size = None  # EPC
    
        long_output_size  = dataset_params['long_term_pred_length']
        short_output_size = dataset_params['short_term_pred_length']
    
        model = create_model(
            model_name=model_name,
            # 장·단기 입력/출력
            long_input_size_num=in_long,
            short_input_size_num=in_short,
            long_output_size=long_output_size,
            short_output_size=short_output_size,
            # 공통
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            # 아이콘(UMass만 유효)
            icon_vocab_size=icon_vocab_size,
            icon_embed_dim=icon_embed_dim,
            # 참고: 길이(모델이 필요로 하면)
            long_term_length=dataset_params['long_term_length'],
            short_term_length=dataset_params['short_term_length']
        )

        suffix = path_suffix or ("umass" if icon_vocab_size else "epc")
        model_path = './tmp/{}_{}_long{}_short{}_pred{}_hs{}_nl{}_a{}_b{}_{}.pth'.format(
            model_name, suffix,
            dataset_params['long_term_length'], dataset_params['short_term_length'],
            dataset_params['long_term_pred_length'],
            hidden_size, num_layers, mse_alpha, mse_beta, feature_num
        )
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        print("Model path:", model_path)

        # ---------------------------
        # 로드 or 학습
        # ---------------------------
        if os.path.exists(model_path):
            print(f"Loading the pre-trained {model_name} ...")
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        else:
            print(f"{model_name} not found. Training a new one (long-term)...")
            train_for_long_term_forecast(
                model=model,
                model_name=model_name,
                train_data=train_data,          # 전역 (EPC: (XL, XS) / UMass: ((XnL, XiL), (XnS, XiS)))
                train_targets=train_targets,    # (yL, yS)
                eval_data=eval_data,            # 구조 동일
                eval_targets=eval_targets,      # (yL, yS)
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience,
                oversample_eval=oversample_eval,  # 평가시 오버샘플 여부
                alpha=mse_alpha,
                beta=mse_beta
            )

        
    else:
        # EPC: train_data = Xn
        # UMass: train_data = (Xn, Xi)
        is_umass = isinstance(train_data, (tuple, list))
    
        if is_umass:
            Xn, Xi = train_data
            in_single = Xn.shape[2]
            icon_vocab_size = int(Xi.max().item()) + 1
        else:
            Xn = train_data
            in_single = Xn.shape[2]
            icon_vocab_size = None
    
        output_size_single = dataset_params.get('prediction_length', 24)
    
        model = create_model(
            model_name=model_name,
            input_size_num=in_single,
            output_size=output_size_single,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            icon_vocab_size=icon_vocab_size,
            icon_embed_dim=icon_embed_dim
        )

        suffix = path_suffix or ("umass" if icon_vocab_size else "epc")
        model_path = './tmp/{}_{}_seq{}_pred{}_hs{}_nl{}_{}.pth'.format(
            model_name, suffix,
            dataset_params['sequence_length'], dataset_params['prediction_length'],
            hidden_size, num_layers, feature_num
        )
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        print("Model path:", model_path)
    
        if os.path.exists(model_path):
            print(f"Loading the pre-trained {model_name} ...")
            model.load_state_dict(torch.load(model_path, map_location='cpu'))
            model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
        else:
            print(f"{model_name} not found. Training a new one...")
            train_for_short_term_forecast(
                model=model,
                model_name=model_name,
                train_sequences=train_data,
                train_targets=train_targets,
                eval_sequences=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience
            )

    return model, model_path

In [6]:
n = 7

if is_long_term_forecast:
    train_data, train_targets, eval_data, eval_targets, selected_features, target_scaler = load_dataset(
        params=dataset_params,
        dataset_type=dataset_name,
        file_path=file_path,
        is_long_term_forecast=is_long_term_forecast,
        target_name=target_name,
        categorical_features=('icon',),
        filtered_features={'long':base_features[:n], 'short':base_features[:n]}
    )
    feature_num = len(selected_features['long'])

else:
    train_data, train_targets, eval_data, eval_targets, selected_features, target_scaler = load_dataset(
        params=dataset_params,
        dataset_type=dataset_name,
        file_path=file_path,
        is_long_term_forecast=is_long_term_forecast,
        target_name=target_name,
        categorical_features=('icon',),
        filtered_features={'single':base_features[:n]}
    )
    feature_num = len(selected_features['single'])


print(feature_num)
print(selected_features)
print(train_data[0].shape)

12
{'single': ['use_total', 'use_m2', 'FurnaceHRV_m2', 'CellarOutlets_m2', 'WashingMachine_m2', 'FridgeRange_m2', 'sin_hour', 'cos_hour', 'sin_day', 'cos_day', 'sin_month', 'cos_month']}
torch.Size([20446, 720, 12])


In [11]:
# model_params = {
#     'model_name'  : 'GRU',
#     'hidden_size' : 512,
#     'num_layers'  : 3,
#     'dropout'     : 0.3,
#     'num_epochs'  : 1,
#     'batch_size'  : 128,
#     'learning_rate': 0.001,
#     'patience'    : 15,
#     'mse_decay'   : True
# }

model_params = {
    'model_name'  : 'Transformer',
    'hidden_size' : 192,
    'num_layers'  : 4,
    'dropout'     : 0.3,
    'num_epochs'  : 1,
    'batch_size'  : 128,
    'learning_rate': 0.001,
    'patience'    : 15,
    'mse_decay'   : False
}

# GPU 메모리 초기화 (선택)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# 시작 시간
torch.cuda.synchronize()
t0 = time.perf_counter()

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


#########
model, model_path = load_trained_model(
    params=model_params,
    is_long_term_forecast=is_long_term_forecast,
    oversample_eval=True  # 평가셋 오버샘플 여부(기존 정책)
)

#########

# 종료 시간
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

# 메모리 사용량 확인 (선택)
peak_mem = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else None

print(f"Training Time (1 epoch): {elapsed:.2f} s")
if peak_mem:
    print(f"Peak GPU Memory: {peak_mem:.1f} MB")


total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params/1e6:.3f} M")


Model path: ./tmp/Transformer_umass_seq720_pred24_hs192_nl4_12.pth
Transformer not found. Training a new one...


Epoch 001/1 | Train 0.040727 | Val 0.012709
[Short] ✓ Epoch 1 improved → saved (val=0.01270872, train=0.04072727)
Training Time (1 epoch): 3.62 s
Peak GPU Memory: 516.1 MB
Trainable parameters: 1.493 M


In [12]:
model, model_path = load_trained_model(
    params=model_params,
    is_long_term_forecast=is_long_term_forecast,
    oversample_eval=True  # 평가셋 오버샘플 여부(기존 정책)
)


# GPU 동기화 (정확한 측정 위해)
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()

batch_size = 1000

if is_long_term_forecast:
    # -------- Long-term forecast 평가 --------
    results = evaluate_for_long_term_forecast(
        model=model,
        eval_data=eval_data,
        eval_targets=eval_targets,
        model_name=model_params['model_name'],
        batch_size=batch_size,
        oversample_eval=False,     # 기존 정책 유지
        # target_scaler=target_scaler
    )
else:
    # -------- Short-term forecast 평가 --------
    # EPC: eval_data = X, eval_targets = y
    # UMass: eval_data = (Xn, Xi), eval_targets = y
    results = evaluate_for_short_term_forecast(
        model=model,
        eval_sequences=eval_data,     # EPC는 X, UMass는 (Xn, Xi)
        eval_targets=eval_targets,
        model_name=model_params['model_name'],
        batch_size=batch_size,
        # target_scaler=target_scaler
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

print(f"Evaluation results: {results}")
print(f"Inference time: {elapsed:.3f} s")

/tmp/ipykernel_1353605/3658061702.py:288: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location='cpu'))


Model path: ./tmp/Transformer_umass_seq720_pred24_hs192_nl4_12.pth
Loading the pre-trained Transformer ...
R² Score: 0.6004
Adjusted R²: 0.5995
SMAPE: 47.30
MASE: 1.3597
Evaluation results: {'R2': 0.6004483699798584, 'Adjusted R2': 0.5995080641237608, 'SMAPE': np.float32(47.304478), 'MASE': np.float32(1.3596574)}
Inference time: 0.303 s
